# Requesting data from the remote sensors

This example shows, how the actual data can be read (i.e. _pulled_) from the remote sensors. It is also possible for the sensors to _push_ data e.g. to an edge node. The remote sensor board is a Raspberry Pi 4 board computer with two DHT22 sensors. We can query our remote sensors via the REST API, which allows us to query a single sensor `dht1` or `dht2`. The Python code is quite straightforward and can be executed in the following code cell. The remote sensor unit operates as a _data producer_ and the script below _consumes_ the data.


```
# Read data from dwbl.dclabra.fi
import requests
resp = requests.get('https://dwbl.dclabra.fi/api/v1/dht1')
print(resp.json())
```

Execute the code cell below to see the results.

In [ ]:
# Read data from dwbl.dclabra.fi
import requests

resp = requests.get('https://dwbl.dclabra.fi/api/v1/dht1')
print(resp.json())



If the query was successful, the result should look something like

`{'humidity': 40.5, 'sensor_id': 'dht1', 'sensor_type': 'dht22', 'temperature': 28.200000762939453}`

The response is provided in JSON format with four key-value pairs

```json
{
  "humidity": 39.5,
  "sensor_id": "dht1",
  "sensor_type": "dht22",
  "temperature": 28.299999237060547
}
```

The keys are presented in alphabetical order:

- **humidity** : humidity value as float
- **sensor_id** : Identifier of the sensor, provided as last parameter in HTTP request
- **sensor_type** : Type of the sensor (DHT22)
- **temperature** : Temperature of the current measurement

One extra parameter could be the timestamp of the measurement, formatted e.g. in Unix epoch time. That would enable us the [_time series_](https://en.wikipedia.org/wiki/Time_series) data. The timestamp can also be added by the client (i.e. the data consumer), once the measured data is received.

For curiosity, it would be great to understand, how the previous data is received from the DHT22 sensors connected to the Raspberry Pi hardware. Let's take a look at the internals of the measuring equipment. This example allows us to utilize a Raspberry Pi hardware with the possibility to read the humidity and temperature values provided by the DHT22 sensors connected to Raspberry Pi. Below you'll find the architecture of the edge node (Raspberry Pi).

![./Kuvat/sensor-node-architecture.svg](https://staging-iot.web.app/chapters/images/sensor-node-architecture.svg)
Picture 3. Edge node architecture.

The interaction with DHT22 sensors is done with the ADAFruit library, which is now updated to [Adafruit IO](https://github.com/adafruit/Adafruit_IO_Python). The corresponding code `sensors.py` is found below:

```python
import Adafruit_DHT

def get_dht1():
  return Adafruit_DHT.read_retry(Adafruit_DHT.DHT22, 17)

def get_dht2():
  return Adafruit_DHT.read_retry(Adafruit_DHT.DHT22, 18)
```

The sensor type and it's GPIO port are provided as parameters for the `read_retry(sensor, gpio)` function. Next we shall implement a simple REST server with [Flask](https://flask.palletsprojects.com/en/1.1.x/). REST (Representational State Transfer) is a simple yet effective implementation using stateless [HTTP methods](https://restfulapi.net/http-methods/) (POST, GET, PUT, PATCH, and DELETE) to operate with the server. It is used in many kinds of server implementations to interchange data and operate between client and other software modules. Below you'll find a simple webserver implemented in Python.

`webserver.py`

```python
from flask import Flask, jsonify
import sensors

app = Flask(__name__)

# Access to server root
@app.route('/')
def index():
  return "~ Welcome to KAMK sensor hub ~"

# Access to sensors
@app.route('/api/v1/<string:sensor_id>', methods=['GET'])
def get_sensor(sensor_id):

  if(len(sensor_id) == 0 or sensor_id == ' '):
    return jsonify({'sensor_id': 'not found'})

  if sensor_id == "dht1":
    humidity, temperature = sensors.get_dht1()
  elif sensor_id == "dht2":
    humidity, temperature = sensors.get_dht2()

  return jsonify({'sensor_id': sensor_id, 'sensor_type': 'dht22', 'humidity': humidity, 'temperature': temperature})

if __name__ == "__main__":
  app.run(host='0.0.0.0', debug=True, port=8080)
```

Once started from command line with `python3 webserver.py` the Flask server, the server provides two routes for the HTTP GET requests

| Resource      | Explanation                                                          |
| ------------- | -------------------------------------------------------------------- |
| /             | root URI (index) returning text \~ Welcome to my simple sensor node \~ |
| /api/v1/dht1  | Resource endpoint for sensor 1                                       |
| /api/v1/dht2  | Resource endpoint for sensor 2                                       |

This actual example is run on a plain Raspberry Pi 4 in <https://dwbl.dclabra.fi/>.